In [ ]:
%matplotlib inline


In [ ]:
import matplotlib.pyplot as plt
import mne

In [ ]:
import hypyp
import hypyp.fnirs as fnirs


In [ ]:
# Setup data
#file_path1 = "/home/pfortin/work/imagine/samba/imagine-data/nirs/2025-09-17/reem_005/reem_005.snirf"
#file_path2 = "/home/pfortin/work/imagine/samba/imagine-data/nirs/2025-09-17/remy_005/remy_005.snirf"
file_path1 = "/home/pfortin/work/imagine/samba/imagine-data/nirs/2025-09-17/reem_005/reem_005_config.hdr"
file_path2 = "/home/pfortin/work/imagine/samba/imagine-data/nirs/2025-09-17/remy_005/remy_005_config.hdr"

In [ ]:
task = hypyp.utils.Task('check', onset_time=0, duration=60)
def get_recordings():
    return [
        fnirs.Recording(subject_label='alice', tasks=[task]).load_file(file_path1, preprocess=False),
        fnirs.Recording(subject_label='bob', tasks=[task]).load_file(file_path2, preprocess=False),
    ]



In [ ]:

recordings = get_recordings()

preprocessor = fnirs.MnePreprocessorAsIs()

for recording in recordings:
    recording.preprocess(preprocessor)

    print(recording.mne_raw.info['subject_info'])





In [ ]:

raw: mne.io.Raw = recordings[0].mne_raw
raw.plot_sensors(show_names=True)


In [ ]:

dyad = fnirs.Dyad(*recordings)

wavelet = hypyp.wavelet.ComplexMorletWavelet(period_range=(0.5, 10))

dyad.compute_wtcs(ch_match='760', wavelet=wavelet)


In [ ]:
for subject_idx, subject_number in enumerate(['1', '2']):
    cwt_key = f"cwt{subject_number}"
    subject_label_key = f"label_s{subject_number}"
    shown = []
    for wtc in dyad.wtcs:
        cwt = getattr(wtc, cwt_key)
        subject_label = getattr(wtc, subject_label_key)
        if cwt.label in shown:
            continue
        shown.append(cwt.label)
        print(cwt.label)
        _ = cwt.plot(title=f"CWT of {subject_label}, {cwt.label}")
        plt.show()
